In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import os


df = pd.read_csv('data/train.csv')
df = df.sort_values('Image').reset_index(drop=True) 

test_size = 0.1
val_size = 0.1

train_val, test = train_test_split(
    df, 
    test_size=test_size, 
    random_state=42, 
    shuffle=True
)

val_relative_size = val_size / (1 - test_size)

train, val = train_test_split(
    train_val, 
    test_size=val_relative_size, 
    random_state=42, 
    shuffle=True
)

# Find all classes seen in the training set
# And make sure "new_whale" is in the allowed classes
train_labels = sorted(list(set(train['Id'].unique())))
if 'new_whale' not in train_labels:
    train_labels.append('new_whale')

id_to_idx = {name: i for i, name in enumerate(train_labels)}
num_classes = len(id_to_idx)

# Replace unseen whales with "new_whale" like in the competition
test.loc[~test['Id'].isin(train_labels), 'Id'] = 'new_whale'
val.loc[~val['Id'].isin(train_labels), 'Id'] = 'new_whale'

print(f"Amount of labels: {len(train_labels)}")
print(f"Train: {len(train)} | Val: {len(val)} | Test: {len(test)}")

Amount of labels: 3788
Train: 7879 | Val: 986 | Test: 985


In [8]:
class WhaleDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        img_name = f"data/train/{self.df.iloc[idx]['Image']}"
        image = Image.open(img_name).convert('RGB')
        label = id_to_idx[self.df.iloc[idx]['Id']]
        
        if self.transform:
            image = self.transform(image)
            
        return image, label

In [10]:
model = models.resnet50(pretrained=True)

# Freeze the backbone
for param in model.parameters():
    param.requires_grad = False

# Replace the FC layer (In ResNet50, it's model.fc)
# model.fc.in_features is 2048
model.fc = nn.Linear(model.fc.in_features, num_classes)

# Speed up if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

c:\Users\Sebastian Munthe\Desktop\MLForCV\ML4CV\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\Sebastian Munthe\Desktop\MLForCV\ML4CV\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [ ]:
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler('cuda')

def save_checkpoint(epoch, model, optimizer, scaler, path="checkpoint.pth"):
    torch.save({
        'epoch': epoch,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scaler_state': scaler.state_dict(),
    }, path)

def train_one_epoch(loader):
    model.train()
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        
        with torch.amp.autocast('cuda'):
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

C:\Users\Sebastian Munthe\AppData\Local\Temp\ipykernel_15056\3518143470.py:3: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
c:\Users\Sebastian Munthe\Desktop\MLForCV\ML4CV\.venv\Lib\site-packages\torch\cuda\amp\grad_scaler.py:31: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  super().__init__(


In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_loader = DataLoader(WhaleDataset(train, transform), batch_size=32, shuffle=True)

# Resume if checkpoint exists
if os.path.exists("checkpoint.pth"):
    ckpt = torch.load("checkpoint.pth")
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    scaler.load_state_dict(ckpt['scaler_state']) 
    print(f"Resumed from epoch {ckpt['epoch']}!")

# Run one epoch and save
train_one_epoch(train_loader)
save_checkpoint(1, model, optimizer, scaler)
print("Finished Stage 1!")

C:\Users\Sebastian Munthe\AppData\Local\Temp\ipykernel_15056\3518143470.py:19: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
c:\Users\Sebastian Munthe\Desktop\MLForCV\ML4CV\.venv\Lib\site-packages\torch\cuda\amp\autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(
